# TradeSense · Component 03 · Stage F · Notebook F1
### Build the labelled sets for TRAINING an order-book manipulation detector, and a shifted validation set to test whether it generalises

**Why.** E2d showed that simple scores see the injected events only weakly (an event's best score is typically in the top 3% of clean time for the order-book change score, top 13% for the activity score; the strict 2 to 10 false alarms per day level finds almost nothing). Single-moment size is not what separates manipulation from normal bursts; the *shape in time and across price levels* is. A model has to learn that shape, so it must be trained for detection with labels. This notebook makes the data for that.

**Set 1 · Training sets (May and June, both symbols).** 300 synthetic events per symbol-day (4 patterns × 3 magnitudes × 25), each with its own random magnitude scale between 0.5 and 2.0 times the E1 setting (so weaker events than E1's "subtle" are included). Pattern 5 is never used here.

**Set 2 · Shifted validation sets (July, both symbols).** 180 events per symbol-day with **all 5 patterns** (pattern 5 *flicker* – an order placed and pulled again and again – is new and never seen in training), magnitude scales 0.6 or 1.5, durations 30% longer, a different random seed from E1. They tell us whether a detector learned the idea or only the shape of our injection. The E1 sets stay as the "standard" validation set.

**Rules.** Stage C and the E1 sets are only read. No test day is opened or injected. Training days are injected only here (declared in Protocol v3, written before any detector is trained). Synthetic results are always reported as synthetic. **No GPU is needed** (CPU, about 30 to 60 minutes, resumable: finished sets are skipped). Output: `TGNN_StageF_v1/`.

## 1 · Settings (nothing to edit)

In [ ]:
import os, gc, sys, json, time, shutil, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
STAGEC_NAME = 'TGNN_StageC_GraphFeatures_v2'; OUT_NAME = 'TGNN_StageF_v1'
SYMS = ['BTCUSDT', 'ETHUSDT']; TRAIN_DAYS = ['2024-05-01', '2024-06-01']; VAL_DAYS = ['2024-07-01']
TRAIN_SPEC = {'n_per_pair': 25, 'patterns': [1, 2, 3, 4], 'scale': ['loguniform', 0.5, 2.0], 'dur_mult': 1.0, 'seed0': 1000}
SHIFT_SPEC = {'n_per_pair': 12, 'patterns': [1, 2, 3, 4, 5], 'scale': ['choice', [0.6, 1.5]], 'dur_mult': 1.3, 'seed0': 7000}
MARGIN = 600; GAP = 600
print('train spec', TRAIN_SPEC, '\nshift spec', SHIFT_SPEC)

Mounted at /content/drive
train spec {'n_per_pair': 25, 'patterns': [1, 2, 3, 4], 'scale': ['loguniform', 0.5, 2.0], 'dur_mult': 1.0, 'seed0': 1000} 
shift spec {'n_per_pair': 12, 'patterns': [1, 2, 3, 4, 5], 'scale': ['choice', [0.6, 1.5]], 'dur_mult': 1.3, 'seed0': 7000}


## 2 · Libraries (written to the Colab working folder, not to Drive)

In [ ]:
%%writefile stage_e_lib.py
"""TradeSense C03 - Stage E library: order-book manipulation INJECTOR (synthetic, labelled as synthetic).
Works on the frozen Stage C tensors (scaled float16). Levels are decoded to quantities, a pattern is added to the quantities of existing price levels
(prices, spread and mid never change), and ONLY the quantity-derived channels are recomputed with the same formulas as Stage C:
  node ch2 log_qty, ch3 d_log_qty, ch4 cum_log_depth, glob ch1 imbalance_top5   (ch1/ch5 distances and glob ch0/ch2 do not depend on quantities).
Intervals outside [start, end+1] of an event are copied untouched. Injection never targets a flagged interval. Training days are never injected."""
import json, os
from pathlib import Path
import numpy as np

K = 25; NN = 50; NCH = 6; GCH = 3; LOGQ, DLQ, CUM = 2, 3, 4; IMB = 1
SCALED_NODE = [1, 2, 3, 4, 5]
PATTERNS = {1: 'spoof', 2: 'layering', 3: 'quote_stuffing', 4: 'liquidity_vacuum'}
MAGS = {1: 'subtle', 2: 'medium', 3: 'easy'}
# magnitude parameters per pattern and level (1 subtle, 2 medium, 3 easy). multipliers are in units of the day-median quantity of that rank
PAR = {1: {1: 3.0, 2: 8.0, 3: 20.0}, 2: {1: 2.0, 2: 5.0, 3: 10.0}, 3: {1: 0.8, 2: 2.0, 3: 4.0}, 4: {1: 0.60, 2: 0.85, 3: 0.95}}
DUR = {1: (10, 100), 2: (20, 150), 3: (30, 100), 4: (10, 50)}          # in 100 ms intervals

# ---------------------------------------------------------------- decode / recompute
def _mu_sd(sc, ch): j = SCALED_NODE.index(ch); return float(sc['node_mean'][j]), float(sc['node_std'][j])
def decode_q(nodes, sc, a, b):
    """quantities of intervals [a,b) from the scaled log_qty channel (float64, >= 0)"""
    m, s = _mu_sd(sc, LOGQ); lq = nodes[a:b, :, LOGQ].astype(np.float64) * s + m; return np.maximum(np.expm1(lq), 0.0)
def encode_channels(q, q_prev, sc, glob_mean, glob_std):
    """scaled (log_qty, d_log_qty, cum_log_depth) of quantities q (m,50) given the quantities of the interval just before (50,) and the scaled imbalance (m,)"""
    lq = np.log1p(q); lq_prev = np.log1p(q_prev)[None]; dlq = lq - np.concatenate([lq_prev, lq[:-1]], 0)
    cum = np.concatenate([np.log1p(np.cumsum(q[:, :K], 1)), np.log1p(np.cumsum(q[:, K:], 1))], 1)
    s5b = q[:, :5].sum(1); s5a = q[:, K:K + 5].sum(1); imb = (s5b - s5a) / np.maximum(s5b + s5a, 1e-12)
    out = []
    for ch, arr in ((LOGQ, lq), (DLQ, dlq), (CUM, cum)): m, s = _mu_sd(sc, ch); out.append((arr - m) / s)
    return out[0], out[1], out[2], (imb - glob_mean) / glob_std

def day_ref(nodes, sc, flags, step=50):
    """per-rank median quantity of the day (sampled every `step` intervals, flag-free intervals only): the unit of injection magnitude"""
    idx = np.flatnonzero(flags[::step] == 0) * step
    m, s = _mu_sd(sc, LOGQ); lq = nodes[idx, :, LOGQ].astype(np.float64) * s + m; return np.median(np.maximum(np.expm1(lq), 0.0), 0)

# ---------------------------------------------------------------- pattern definitions: additive/multiplicative change of q over the event
def pattern_delta(q_span, pat, mag, side, rng, ref, dur):
    """returns the modified quantities q' for the intervals of the event (dur, 50)"""
    q = q_span.copy(); off = 0 if side == 0 else K; par = PAR[pat][mag]; meta = {}
    if pat == 1:                                                   # spoof: one large order a few ranks away from the touch, then cancelled
        r = int(rng.integers(1, 6)); q[:, off + r] += par * ref[off + r]; meta = {'ranks': [r]}
    elif pat == 2:                                                 # layering: 3-6 consecutive ranks, staggered placement, removed together
        L = int(rng.integers(3, 7)); r0 = int(rng.integers(1, 5)); ranks = list(range(r0, r0 + L)); meta = {'ranks': ranks}
        for i, r in enumerate(ranks):
            st = min(i * int(rng.integers(0, 5)), max(dur - 2, 0)); q[st:, off + r] += par * ref[off + r] * rng.uniform(0.7, 1.3)
    elif pat == 3:                                                 # quote stuffing: rapid on/off of quantity on the 10 best ranks
        on = (np.arange(dur) % 2 == 0); q[on, off:off + 10] *= (1.0 + par); meta = {'ranks': list(range(10))}
    elif pat == 4:                                                 # liquidity vacuum: the k best ranks lose most of their quantity
        k = int(rng.integers(3, 9)); q[:, off:off + k] *= (1.0 - par); meta = {'ranks': list(range(k))}
    return q, meta

# ---------------------------------------------------------------- placement
def place_events(flags, win_end, n_events, rng, margin=600, gap=600, durations=DUR, patterns=(1, 2, 3, 4), mags=(1, 2, 3)):
    """random event slots inside long flag-free stretches: [start, end] (inclusive), margin intervals of clean context before and 50 after, at least `gap` intervals between events.
    n_events = events per (pattern, magnitude) pair."""
    n = len(flags); bad = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); taken = np.zeros(n, bool); ev = []
    jobs = [(p, m) for p in patterns for m in mags for _ in range(n_events)]; rng.shuffle(jobs)
    for p, m in jobs:
        dur = int(rng.integers(durations[p][0], durations[p][1] + 1))
        for _try in range(200):
            s = int(rng.integers(margin, n - dur - 60)); e = s + dur - 1
            if bad[e + 52] - bad[s - margin] != 0: continue                # whole stretch must be flag-free
            if taken[max(s - gap, 0):e + gap].any(): continue
            taken[s:e + 1] = True; ev.append({'start': s, 'end': e, 'pattern': p, 'mag': m, 'side': int(rng.integers(0, 2))}); break
    ev.sort(key=lambda d: d['start']); return ev

# ---------------------------------------------------------------- apply
def inject_day(nodes, glob, flags, sc, events, seed):
    """returns copies (nodes2, glob2) and an interval label array; events must come from place_events (flag-free, separated)"""
    nodes2 = nodes.copy(); glob2 = glob.copy(); n = len(flags); pat_arr = np.zeros(n, np.int8); mag_arr = np.zeros(n, np.int8); rng = np.random.default_rng(seed)
    ref = day_ref(nodes, sc, flags); gm = float(sc['glob_mean'][IMB]); gs = float(sc['glob_std'][IMB]); out = []
    for e in events:
        s, t, p, m, side = e['start'], e['end'], e['pattern'], e['mag'], e['side']; dur = t - s + 1
        q = decode_q(nodes, sc, s - 1, t + 2)                          # rows: s-1 (context), s..t (event), t+1 (cancel interval)
        q_ev, meta = pattern_delta(q[1:1 + dur], p, m, side, rng, ref, dur)
        qn = q.copy(); qn[1:1 + dur] = q_ev                              # interval t+1 keeps the original (cancelled) quantities
        lq, dl, cu, im = encode_channels(qn[1:], q[0], sc, gm, gs)
        nodes2[s:t + 2, :, LOGQ] = lq; nodes2[s:t + 2, :, DLQ] = dl; nodes2[s:t + 2, :, CUM] = cu; glob2[s:t + 2, IMB] = im
        pat_arr[s:t + 1] = p; mag_arr[s:t + 1] = m; out.append({**e, **meta, 'pattern_name': PATTERNS[p], 'mag_name': MAGS[m]})
    return nodes2, glob2, pat_arr, mag_arr, out

def build_injected_set(src_dir, dst_dir, sc, n_per_pair, seed, margin=600, gap=600):
    """one Stage C set -> injected set folder with the same layout plus event_pattern.npy, event_mag.npy, events.json. Training sets are refused."""
    src = Path(src_dir); dst = Path(dst_dir); sm = json.loads((src / 'set_summary.json').read_text())
    if sm['split'] == 'train': raise PermissionError('training sets are never injected')
    nodes = np.load(src / 'nodes.npy', mmap_mode='r'); glob = np.load(src / 'glob.npy'); flags = np.load(src / 'flags.npy'); win = np.load(src / 'win_end.npy'); rng = np.random.default_rng(seed)
    ev = place_events(flags, win, n_per_pair, rng, margin=margin, gap=gap); n2, g2, pa, ma, evl = inject_day(np.asarray(nodes), glob, flags, sc, ev, seed + 1)
    dst.mkdir(parents=True, exist_ok=True); np.save(dst / 'nodes.npy', n2.astype(np.float16)); np.save(dst / 'glob.npy', g2.astype(np.float16)); np.save(dst / 'flags.npy', flags); np.save(dst / 'win_end.npy', win)
    np.save(dst / 'event_pattern.npy', pa); np.save(dst / 'event_mag.npy', ma); (dst / 'events.json').write_text(json.dumps(evl, indent=1))
    sm2 = dict(sm); sm2.update({'injected': True, 'synthetic_events': len(evl), 'source_set': str(src.name), 'injection_seed': seed}); (dst / 'set_summary.json').write_text(json.dumps(sm2, indent=1)); return evl

# ---------------------------------------------------------------- tests (synthetic data, no real data involved)
def make_toy(n=6000, seed=0):
    """a tiny consistent Stage-C-like day built with the SAME formulas: random quantities -> scaled channels (so decode/encode must be exact up to float16)"""
    rng = np.random.default_rng(seed); q = np.exp(rng.normal(3.0, 0.8, size=(n, NN))); q = np.where(rng.random((n, NN)) < 0.08, q * rng.uniform(1.5, 4, size=(n, NN)), q)
    for i in range(1, n):
        keep = rng.random(NN) < 0.8; q[i] = np.where(keep, q[i - 1], q[i])                                    # sparse changes
    sc = {'node_mean': [0.0, 3.0, 0.0, 6.0, 0.0], 'node_std': [1.0, 1.3, 0.8, 1.5, 1.0], 'glob_mean': [0.0, 0.0, 0.0], 'glob_std': [1.0, 0.4, 1.0]}
    nodes = np.zeros((n, NN, NCH), np.float32); glob = np.zeros((n, GCH), np.float32); lq, dl, cu, im = encode_channels(q, q[0], sc, 0.0, 0.4)
    nodes[:, :, 0] = np.where(np.arange(NN) < K, -1, 1); nodes[:, :, LOGQ] = lq; nodes[:, :, DLQ] = dl; nodes[:, :, CUM] = cu; nodes[:, :, 1] = rng.normal(size=(n, NN)); nodes[:, :, 5] = rng.normal(size=(n, NN)); glob[:, IMB] = im
    flags = np.zeros(n, np.uint8); flags[:700] = 16; flags[6000:6040] = 8
    return nodes.astype(np.float16), glob.astype(np.float16), flags, sc, q
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    nodes, glob, flags, sc, q_true = make_toy(n=12000); n = len(flags); rng = np.random.default_rng(1)
    # H01 decode gives back the quantities (float16 tolerance)
    qd = decode_q(nodes, sc, 0, n); rel = np.abs(qd - q_true) / np.maximum(q_true, 1.0); add('H01', 'decoded quantities equal the true quantities (float16 tolerance)', float(np.median(rel)) < 2e-3 and float(rel.max()) < 3e-2, f'median {np.median(rel):.1e} max {rel.max():.1e}')
    # H02 re-encoding unchanged quantities reproduces the stored channels
    lq, dl, cu, im = encode_channels(qd[1000:1100], qd[999], sc, 0.0, 0.4); add('H02', 'recomputed log_qty, d_log_qty, cum_log_depth and imbalance of untouched data equal the stored channels', np.abs(lq - nodes[1000:1100, :, LOGQ].astype(np.float64)).max() < 0.02 and np.abs(dl - nodes[1000:1100, :, DLQ].astype(np.float64)).max() < 0.05 and np.abs(cu - nodes[1000:1100, :, CUM].astype(np.float64)).max() < 0.05 and np.abs(im - glob[1000:1100, IMB].astype(np.float64)).max() < 0.05)
    # H03 placement rules
    ev = place_events(flags, None, 3, rng, margin=100, gap=100); ok = len(ev) == 36 and all(e['start'] >= 100 for e in ev); bad = np.concatenate([[0], np.cumsum(flags != 0)])
    ok2 = all(bad[e['end'] + 52] - bad[e['start'] - 100] == 0 for e in ev); ok3 = all(ev[i + 1]['start'] - ev[i]['end'] >= 100 for i in range(len(ev) - 1)); add('H03', '36 events placed (3 per pattern/magnitude), never over flagged intervals, with clean context and gaps', ok and ok2 and ok3, f'{len(ev)} events')
    # H04 untouched outside the event: intervals before start and after end+1 are identical, all pre-existing channels except the four recomputed ones unchanged
    n2, g2, pa, ma, evl = inject_day(nodes, glob, flags, sc, ev, 5); idx = np.arange(n); inside = np.zeros(n, bool)
    for e in ev: inside[e['start']:e['end'] + 2] = True
    same = np.array_equal(n2[~inside], nodes[~inside]) and np.array_equal(g2[~inside], glob[~inside]); other = np.array_equal(n2[:, :, [0, 1, 5]], nodes[:, :, [0, 1, 5]]) and np.array_equal(g2[:, [0, 2]], glob[:, [0, 2]])
    add('H04', 'intervals outside every event are byte-identical; distance, side and spread/mid channels are never changed', same and other)
    # H05 causality: the first changed interval is the event start; nothing before it changes
    first = [int(np.flatnonzero((n2[e['start'] - 100:e['end'] + 2] != nodes[e['start'] - 100:e['end'] + 2]).any((1, 2)))[0]) - 100 for e in ev[:12]]; add('H05', 'the first changed interval of every event is its start (nothing earlier is touched)', all(f == 0 for f in first), first[:6])
    # H06 spoof by hand: extra quantity on the chosen rank equals magnitude x day reference, restored after the event
    ref = day_ref(nodes, sc, flags); sp = [e for e in evl if e['pattern'] == 1][0]; s0, e0 = sp['start'], sp['end']; r = sp['ranks'][0] + (0 if sp['side'] == 0 else K)
    qa = decode_q(n2, sc, s0 - 1, e0 + 3); qb = decode_q(nodes, sc, s0 - 1, e0 + 3); add_q = (qa[1:1 + (e0 - s0 + 1), r] - qb[1:1 + (e0 - s0 + 1), r]); target = PAR[1][sp['mag']] * ref[r]
    add('H06', 'spoof: the added quantity equals the magnitude x day reference of that rank during the whole event and is gone after it', abs(np.median(add_q) - target) / target < 0.03 and abs(qa[-1, r] - qb[-1, r]) / max(qb[-1, r], 1) < 0.03, f'median {np.median(add_q):.1f} target {target:.1f}')
    # H07 vacuum removes quantity, stuffing alternates, layering adds on 3+ ranks
    vac = [e for e in evl if e['pattern'] == 4][0]; qv = decode_q(n2, sc, vac['start'], vac['end'] + 1); qo = decode_q(nodes, sc, vac['start'], vac['end'] + 1); off = 0 if vac['side'] == 0 else K; k = len(vac['ranks'])
    ratio = qv[:, off:off + k].sum() / max(qo[:, off:off + k].sum(), 1e-9); stf = [e for e in evl if e['pattern'] == 3][0]; qs = decode_q(n2, sc, stf['start'], stf['start'] + 6); qso = decode_q(nodes, sc, stf['start'], stf['start'] + 6); o2 = 0 if stf['side'] == 0 else K
    alt = (qs[:, o2] / np.maximum(qso[:, o2], 1e-9)); lay = [e for e in evl if e['pattern'] == 2][0]
    add('H07', 'vacuum keeps the expected share of quantity, stuffing alternates on/off, layering touches 3 to 6 ranks', abs(ratio - (1 - PAR[4][vac['mag']])) < 0.06 and alt[0] > 1.5 and abs(alt[1] - 1) < 0.05 and 3 <= len(lay['ranks']) <= 6, f'vacuum kept {ratio:.2f}; stuffing {alt[:3].round(2).tolist()}')
    # H08 labels, determinism, training refusal
    lab_ok = all((pa[e['start']:e['end'] + 1] == e['pattern']).all() and (ma[e['start']:e['end'] + 1] == e['mag']).all() for e in evl) and int((pa > 0).sum()) == sum(e['end'] - e['start'] + 1 for e in evl)
    n3, g3, *_ = inject_day(nodes, glob, flags, sc, ev, 5); det = np.array_equal(n2, n3) and np.array_equal(g2, g3)
    d = tmp / 'src_train'; d.mkdir(parents=True, exist_ok=True); (d / 'set_summary.json').write_text(json.dumps({'split': 'train'})); refused = False
    try: build_injected_set(d, tmp / 'dst', sc, 1, 0)
    except PermissionError: refused = True
    add('H08', 'interval labels equal the events, the same seed gives identical tensors, training sets are refused', lab_ok and det and refused)
    # H09 end-to-end on a folder: val-like set written, readable by the D1 loader layout, injected count as expected
    sd = tmp / 'src_val'; sd.mkdir(parents=True, exist_ok=True); np.save(sd / 'nodes.npy', nodes); np.save(sd / 'glob.npy', glob); np.save(sd / 'flags.npy', flags); np.save(sd / 'win_end.npy', np.arange(700, n - 2, 5).astype(np.int32)); (sd / 'set_summary.json').write_text(json.dumps({'split': 'val', 'n_intervals': n}))
    evx = build_injected_set(sd, tmp / 'dst_val', sc, 2, 3, margin=100, gap=100); fl = sorted(p.name for p in (tmp / 'dst_val').iterdir()); add('H09', 'a validation set is written with labels and events; summary says injected', len(evx) == 24 and 'event_pattern.npy' in fl and json.loads((tmp / 'dst_val' / 'set_summary.json').read_text())['injected'] is True, fl)
    return out

Writing stage_e_lib.py


In [ ]:
%%writefile stage_f_lib.py
"""TradeSense C03 - Stage F library: injected sets for TRAINING a detector (train days) and for testing it on shifted data (validation days).
Builds on the frozen Stage E injector (stage_e_lib, unchanged). Additions: a per-event magnitude scale, a duration multiplier, and one extra pattern
(5 = flicker: an order that is placed and pulled repeatedly) that is used ONLY in the shifted validation sets, never in training.
Test days are refused. Everything produced here is synthetic and labelled as such."""
import json
from pathlib import Path
import numpy as np
import stage_e_lib as E

NAMES = {**E.PATTERNS, 5: 'flicker'}
FLICKER_DUR = (60, 300)

def scaled_par(p, m, scale):
    b = E.PAR[p][m]
    return float(np.clip(b * scale, 0.30, 0.98)) if p == 4 else float(b * scale)

def delta_flicker(q_span, mag, side, rng, ref, dur, scale):
    q = q_span.copy(); off = 0 if side == 0 else E.K; par = E.PAR[1][mag] * scale; r = int(rng.integers(1, 6)); on = np.zeros(dur, bool); i = 0; state = True
    while i < dur:
        L = int(rng.integers(5, 16)) if state else int(rng.integers(3, 9)); on[i:i + L] = state; i += L; state = not state
    q[on, off + r] += par * ref[off + r]; return q, {'ranks': [r]}

def place_events_f(flags, n_per_pair, rng, patterns, scale_spec, dur_mult=1.0, margin=600, gap=600):
    """E.place_events with duration multiplier and a magnitude scale per event. scale_spec: ('loguniform', lo, hi) or ('choice', [..])"""
    dur = {p: (max(int(a * dur_mult), 3), max(int(b * dur_mult), 4)) for p, (a, b) in {**E.DUR, 5: FLICKER_DUR}.items()}
    ev = E.place_events(flags, None, n_per_pair, rng, margin=margin, gap=gap, durations=dur, patterns=tuple(patterns))
    for e in ev:
        e['scale'] = float(np.exp(rng.uniform(np.log(scale_spec[1]), np.log(scale_spec[2])))) if scale_spec[0] == 'loguniform' else float(rng.choice(scale_spec[1]))
    return ev

def inject_day_f(nodes, glob, flags, sc, events, seed):
    nodes2 = nodes.copy(); glob2 = glob.copy(); n = len(flags); pat_arr = np.zeros(n, np.int8); mag_arr = np.zeros(n, np.int8); rng = np.random.default_rng(seed)
    ref = E.day_ref(nodes, sc, flags); gm = float(sc['glob_mean'][E.IMB]); gs = float(sc['glob_std'][E.IMB]); out = []; saved = {p: dict(E.PAR[p]) for p in E.PAR}
    try:
        for e in events:
            s, t, p, m, side, scale = e['start'], e['end'], e['pattern'], e['mag'], e['side'], e['scale']; dur = t - s + 1
            q = E.decode_q(nodes, sc, s - 1, t + 2)
            if p == 5: q_ev, meta = delta_flicker(q[1:1 + dur], m, side, rng, ref, dur, scale)
            else:
                E.PAR[p][m] = scaled_par(p, m, scale); q_ev, meta = E.pattern_delta(q[1:1 + dur], p, m, side, rng, ref, dur); E.PAR[p][m] = saved[p][m]
            qn = q.copy(); qn[1:1 + dur] = q_ev
            lq, dl, cu, im = E.encode_channels(qn[1:], q[0], sc, gm, gs)
            nodes2[s:t + 2, :, E.LOGQ] = lq; nodes2[s:t + 2, :, E.DLQ] = dl; nodes2[s:t + 2, :, E.CUM] = cu; glob2[s:t + 2, E.IMB] = im
            pat_arr[s:t + 1] = p; mag_arr[s:t + 1] = m; out.append({**e, **meta, 'pattern_name': NAMES[p], 'mag_name': E.MAGS[m]})
    finally:
        for p in saved: E.PAR[p].update(saved[p])
    return nodes2, glob2, pat_arr, mag_arr, out

def build_set_f(src_dir, dst_dir, sc, n_per_pair, seed, patterns, scale_spec, dur_mult=1.0, margin=600, gap=600, allow_train=False):
    src = Path(src_dir); dst = Path(dst_dir); sm = json.loads((src / 'set_summary.json').read_text())
    if sm['split'] == 'test': raise PermissionError('test days are never injected here')
    if sm['split'] == 'train' and not allow_train: raise PermissionError('training sets need allow_train=True')
    if sm['split'] == 'val' and allow_train: raise PermissionError('allow_train is only for training sets')
    if sm['split'] == 'train' and 5 in patterns: raise PermissionError('pattern 5 (flicker) is held out and never used in training')
    nodes = np.load(src / 'nodes.npy', mmap_mode='r'); glob = np.load(src / 'glob.npy'); flags = np.load(src / 'flags.npy'); win = np.load(src / 'win_end.npy'); rng = np.random.default_rng(seed)
    ev = place_events_f(flags, n_per_pair, rng, patterns, scale_spec, dur_mult, margin, gap); n2, g2, pa, ma, evl = inject_day_f(np.asarray(nodes), glob, flags, sc, ev, seed + 1)
    dst.mkdir(parents=True, exist_ok=True); np.save(dst / 'nodes.npy', n2.astype(np.float16)); np.save(dst / 'glob.npy', g2.astype(np.float16)); np.save(dst / 'flags.npy', flags); np.save(dst / 'win_end.npy', win)
    np.save(dst / 'event_pattern.npy', pa); np.save(dst / 'event_mag.npy', ma); (dst / 'events.json').write_text(json.dumps(evl, indent=1))
    sm2 = dict(sm); sm2.update({'injected': True, 'synthetic_events': len(evl), 'source_set': src.name, 'injection_seed': seed, 'patterns': list(patterns), 'dur_mult': dur_mult}); (dst / 'set_summary.json').write_text(json.dumps(sm2, indent=1)); return evl

# ---------------------------------------------------------------- tests
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    nodes, glob, flags, sc, q = E.make_toy(n=30000); n = len(flags); rng = np.random.default_rng(2)
    ev = place_events_f(flags, 1, rng, (1, 2, 3, 4), ('choice', [1.0]), margin=100, gap=100)
    add('L01', 'placement gives 12 events with a scale each; durations follow the multiplier', len(ev) == 12 and all('scale' in e for e in ev) and all(E.DUR[e['pattern']][0] <= e['end'] - e['start'] + 1 <= E.DUR[e['pattern']][1] for e in ev))
    ev2 = place_events_f(flags, 1, np.random.default_rng(2), (1, 2, 3, 4), ('choice', [1.0]), dur_mult=2.0, margin=100, gap=100); add('L02', 'duration multiplier 2 makes events longer on average', np.mean([e['end'] - e['start'] for e in ev2]) > 1.4 * np.mean([e['end'] - e['start'] for e in ev]))
    spo = [dict(e) for e in ev if e['pattern'] == 1][:1]; res = []
    for sca in (1.0, 2.0):
        for e in spo: e['scale'] = sca
        n2, g2, pa, ma, evl = inject_day_f(nodes, glob, flags, sc, spo, 9); e0 = evl[0]; r_ = e0['ranks'][0] + (0 if e0['side'] == 0 else E.K)
        qa = E.decode_q(n2, sc, e0['start'], e0['end'] + 1)[:, r_]; qb = E.decode_q(nodes, sc, e0['start'], e0['end'] + 1)[:, r_]; res.append(float(np.median(qa - qb)))
    add('L03', 'spoof with scale 2 adds about twice the extra quantity of scale 1', 1.7 < res[1] / max(res[0], 1e-9) < 2.3, f'ratio {res[1] / max(res[0], 1e-9):.2f}')
    fl = [{'start': 2000, 'end': 2199, 'pattern': 5, 'mag': 3, 'side': 0, 'scale': 1.0}]; n3, g3, pa3, ma3, ev3 = inject_day_f(nodes, glob, flags, sc, fl, 4); r5 = ev3[0]['ranks'][0]
    d = E.decode_q(n3, sc, 2000, 2200)[:, r5] - E.decode_q(nodes, sc, 2000, 2200)[:, r5]; on = d > 0.5 * d.max(); sw = int((np.diff(on.astype(int)) != 0).sum())
    add('L04', 'flicker switches on and off several times and the extra quantity is zero when off', sw >= 6 and float(np.abs(d[~on]).max()) < 0.2 * d.max(), f'{sw} switches')
    same = np.array_equal(np.delete(n3, np.arange(2000, 2202), 0), np.delete(nodes, np.arange(2000, 2202), 0)); add('L05', 'nothing outside the event changes', same)
    import json as _j
    def mkset(split):
        d = Path(tmp) / f'src_{split}'; d.mkdir(parents=True, exist_ok=True); np.save(d / 'nodes.npy', nodes); np.save(d / 'glob.npy', glob); np.save(d / 'flags.npy', flags); np.save(d / 'win_end.npy', np.arange(700, n - 10, 3).astype(np.int32))
        (d / 'set_summary.json').write_text(_j.dumps({'split': split, 'n_intervals': n})); return d
    ok = {}
    for split, kw in (('test', {}), ('train', {}), ('val', {'allow_train': True})):
        try: build_set_f(mkset(split), Path(tmp) / f'dst_{split}', sc, 1, 1, (1, 2), ('choice', [1.0]), margin=100, gap=100, **kw); ok[split + str(kw)] = False
        except PermissionError: ok[split + str(kw)] = True
    try: build_set_f(mkset('train'), Path(tmp) / 'dst_t5', sc, 1, 1, (1, 5), ('choice', [1.0]), margin=100, gap=100, allow_train=True); ok['flick_train'] = False
    except PermissionError: ok['flick_train'] = True
    add('L06', 'test days refused; training needs allow_train; validation cannot use allow_train; flicker refused in training', all(ok.values()), ok)
    evt = build_set_f(mkset('train'), Path(tmp) / 'dst_train_ok', sc, 1, 5, (1, 2, 3, 4), ('loguniform', 0.5, 2.0), margin=100, gap=100, allow_train=True); pa = np.load(Path(tmp) / 'dst_train_ok' / 'event_pattern.npy')
    add('L07', 'label array marks exactly the event intervals and the scales lie in range', int((pa > 0).sum()) == sum(e['end'] - e['start'] + 1 for e in evt) and all(0.5 <= e['scale'] <= 2.0 for e in evt) and len({round(e['scale'], 3) for e in evt}) > 6)
    a = build_set_f(mkset('val'), Path(tmp) / 'v1', sc, 1, 7, (1, 2, 3, 4, 5), ('choice', [0.6, 1.5]), margin=100, gap=100); b = build_set_f(mkset('val'), Path(tmp) / 'v2', sc, 1, 7, (1, 2, 3, 4, 5), ('choice', [0.6, 1.5]), margin=100, gap=100)
    add('L08', 'same seed gives identical sets; shifted validation set contains all five patterns', np.array_equal(np.load(Path(tmp) / 'v1' / 'nodes.npy'), np.load(Path(tmp) / 'v2' / 'nodes.npy')) and {e['pattern'] for e in a} == {1, 2, 3, 4, 5})
    return out

Writing stage_f_lib.py


## 3 · Safety checks and frozen configuration

In [ ]:
sys.path.insert(0, str(Path.cwd())); import stage_e_lib as E, stage_f_lib as F
def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
LIB_SHA = {'stage_e_lib': _sha('stage_e_lib.py'), 'stage_f_lib': _sha('stage_f_lib.py')}
SC_DIR = BASE / STAGEC_NAME; assert (SC_DIR / 'sets').is_dir(), f'Stage C folder not found: {SC_DIR}'
sc_file = json.loads((SC_DIR / 'config' / 'scaler_train_only.json').read_text()); SC = sc_file['scaler']; SC_SHA = sc_file['sha256']
TKEYS = [f'{s}_{d}' for d in TRAIN_DAYS for s in SYMS]; VKEYS = [f'{s}_{d}' for d in VAL_DAYS for s in SYMS]
for k, sp in [(k, 'train') for k in TKEYS] + [(k, 'val') for k in VKEYS]:
    sm = json.loads((SC_DIR / 'sets' / k / 'set_summary.json').read_text()); assert sm['split'] == sp, f'{k} is not a {sp} set'
OUT = BASE / OUT_NAME; (OUT / 'config').mkdir(parents=True, exist_ok=True); (OUT / 'reports').mkdir(exist_ok=True); (OUT / 'train_injected' / 'sets').mkdir(parents=True, exist_ok=True); (OUT / 'val_shift' / 'sets').mkdir(parents=True, exist_ok=True)
FROZEN = {'stagec': STAGEC_NAME, 'scaler_sha256': SC_SHA, 'train_spec': TRAIN_SPEC, 'shift_spec': SHIFT_SPEC, 'margin': MARGIN, 'gap': GAP, 'par': {str(k): v for k, v in E.PAR.items()}, 'dur': {str(k): v for k, v in E.DUR.items()}, 'flicker_dur': F.FLICKER_DUR}
def _canon(o): return hashlib.sha256(json.dumps(o, sort_keys=True).encode()).hexdigest()
fp = OUT / 'config' / 'frozen_config_f1.json'
if not fp.exists(): fp.write_text(json.dumps({'config': FROZEN, 'library_sha256': LIB_SHA, 'sha256': _canon(FROZEN)}, indent=1)); print('frozen config written')
else: assert json.loads(fp.read_text())['sha256'] == _canon(FROZEN), 'settings differ from the saved frozen configuration; do not change them while resuming'; print('frozen config matches')
print('scaler', SC_SHA[:16], '| train', TKEYS, '| shifted val', VKEYS)

frozen config written
scaler 2bfe9d4985698ba2 | train ['BTCUSDT_2024-05-01', 'ETHUSDT_2024-05-01', 'BTCUSDT_2024-06-01', 'ETHUSDT_2024-06-01'] | shifted val ['BTCUSDT_2024-07-01', 'ETHUSDT_2024-07-01']


## 4 · Tests

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as t_: ER = E.unit_tests(t_) + F.unit_tests(t_)
display(pd.DataFrame(ER, columns=['id', 'test', 'pass', 'detail'])); assert all(r[2] for r in ER), 'A test failed: stop here and send me this table.'
print(f'tests: {sum(r[2] for r in ER)}/{len(ER)} passed')

,id,test,pass,detail
0,H01,decoded quantities equal the true quantities (...,True,median 7.6e-05 max 1.3e-03
1,H02,"recomputed log_qty, d_log_qty, cum_log_depth a...",True,
2,H03,"36 events placed (3 per pattern/magnitude), ne...",True,36 events
3,H04,intervals outside every event are byte-identic...,True,
4,H05,the first changed interval of every event is i...,True,"[0, 0, 0, 0, 0, 0]"
5,H06,spoof: the added quantity equals the magnitude...,True,median 391.7 target 391.8
6,H07,"vacuum keeps the expected share of quantity, s...",True,"vacuum kept 0.40; stuffing [3.0, 1.0, 3.0]"
7,H08,"interval labels equal the events, the same see...",True,
8,H09,a validation set is written with labels and ev...,True,"['event_mag.npy', 'event_pattern.npy', 'events..."
9,L01,placement gives 12 events with a scale each; d...,True,


tests: 17/17 passed


## 5 · Build the training sets and the shifted validation sets (resumable)

In [ ]:
t0 = time.time(); EV = {}
def build(group, keys, spec, folder, allow_train):
    for i, k in enumerate(keys):
        dst = OUT / folder / 'sets' / k; key = f'{folder}/{k}'
        if (dst / 'set_summary.json').exists() and (dst / 'events.json').exists(): EV[key] = json.loads((dst / 'events.json').read_text()); print(key, 'already built:', len(EV[key]), 'events'); continue
        if dst.exists(): shutil.rmtree(dst)
        sc_spec = tuple(spec['scale']); EV[key] = F.build_set_f(SC_DIR / 'sets' / k, dst, SC, spec['n_per_pair'], spec['seed0'] + i, tuple(spec['patterns']), sc_spec, spec['dur_mult'], MARGIN, GAP, allow_train=allow_train)
        print(f'{key}: {len(EV[key])} events written ({time.time()-t0:.0f} s)'); gc.collect()
build('train', TKEYS, TRAIN_SPEC, 'train_injected', True)
build('shift', VKEYS, SHIFT_SPEC, 'val_shift', False)

train_injected/BTCUSDT_2024-05-01: 300 events written (14 s)
train_injected/ETHUSDT_2024-05-01: 300 events written (29 s)
train_injected/BTCUSDT_2024-06-01: 300 events written (54 s)
train_injected/ETHUSDT_2024-06-01: 300 events written (69 s)
val_shift/BTCUSDT_2024-07-01: 180 events written (87 s)
val_shift/ETHUSDT_2024-07-01: 180 events written (103 s)


## 6 · Checks on the real sets

In [ ]:
RT = []
def R(i, nm, ok, det=''): RT.append((i, nm, bool(ok), str(det)))
rows = []; ok_same = True; ok_clean = True; CH = 200_000
for folder, keys in (('train_injected', TKEYS), ('val_shift', VKEYS)):
    for k in keys:
        src = SC_DIR / 'sets' / k; dst = OUT / folder / 'sets' / k; n0 = np.load(src / 'nodes.npy', mmap_mode='r'); n1 = np.load(dst / 'nodes.npy', mmap_mode='r'); g0 = np.load(src / 'glob.npy'); g1 = np.load(dst / 'glob.npy')
        pa = np.load(dst / 'event_pattern.npy'); fl = np.load(dst / 'flags.npy'); touched = np.zeros(len(pa), bool); evl = EV[f'{folder}/{k}']
        for e in evl: touched[e['start']:e['end'] + 2] = True
        outside = 0
        for a in range(0, len(pa), CH):
            b = min(a + CH, len(pa)); dif = (np.asarray(n0[a:b]) != np.asarray(n1[a:b])).any((1, 2)) | (g0[a:b] != g1[a:b]).any(1); outside += int((dif & ~touched[a:b]).sum())
        ok_same &= outside == 0; ok_clean &= not (fl[touched] != 0).any()
        rows.append({'set': f'{folder}/{k}', 'events': len(evl), 'event intervals': int((pa > 0).sum()), 'share of day %': 100 * float((pa > 0).mean()), 'patterns': sorted({e['pattern'] for e in evl}), 'scale min': min(e['scale'] for e in evl), 'scale max': max(e['scale'] for e in evl), 'changed outside events': outside})
display(pd.DataFrame(rows).round(3))
R('S01', 'only intervals inside events differ from Stage C', ok_same)
R('S02', 'no event touches a flagged interval', ok_clean)
R('S03', 'training sets contain patterns 1-4 only; shifted validation sets contain all five patterns', all(set(r['patterns']) == {1, 2, 3, 4} for r in rows if r['set'].startswith('train')) and all(set(r['patterns']) == {1, 2, 3, 4, 5} for r in rows if r['set'].startswith('val')))
R('S04', 'only train and validation days were read; no test day opened', all(k.split('_', 1)[1] in TRAIN_DAYS + VAL_DAYS for k in TKEYS + VKEYS))
R('S05', 'values finite (sampled)', all(np.isfinite(np.asarray(np.load(OUT / f / 'sets' / k / 'nodes.npy', mmap_mode='r')[::20])).all() for f, ks in (('train_injected', TKEYS), ('val_shift', VKEYS)) for k in ks))
R('S06', 'the E1 standard validation sets are untouched and present', all((BASE / 'TGNN_StageE_v1' / 'val_injected' / 'sets' / k / 'events.json').exists() for k in VKEYS))
display(pd.DataFrame(RT, columns=['id', 'test', 'pass', 'detail']))

,set,events,event intervals,share of day %,patterns,scale min,scale max,changed outside events
0,train_injected/BTCUSDT_2024-05-01,300,16632,1.925,"[1, 2, 3, 4]",0.502,1.999,0
1,train_injected/ETHUSDT_2024-05-01,300,18103,2.095,"[1, 2, 3, 4]",0.503,1.992,0
2,train_injected/BTCUSDT_2024-06-01,300,17478,2.023,"[1, 2, 3, 4]",0.504,1.990,0
3,train_injected/ETHUSDT_2024-06-01,300,17454,2.020,"[1, 2, 3, 4]",0.500,1.999,0
4,val_shift/BTCUSDT_2024-07-01,180,18189,2.105,"[1, 2, 3, 4, 5]",0.600,1.500,0
5,val_shift/ETHUSDT_2024-07-01,180,19445,2.251,"[1, 2, 3, 4, 5]",0.600,1.500,0


,id,test,pass,detail
0,S01,only intervals inside events differ from Stage C,True,
1,S02,no event touches a flagged interval,True,
2,S03,training sets contain patterns 1-4 only; shift...,True,
3,S04,only train and validation days were read; no t...,True,
4,S05,values finite (sampled),True,
5,S06,the E1 standard validation sets are untouched ...,True,


## 7 · Saved report (never overwritten) and the SHARE BLOCK

In [ ]:
ALL = [(r[0], r[1], r[2], r[3]) for r in ER] + RT; ok = all(r[2] for r in ALL); VERDICT = 'F1 COMPLETE' if ok else 'F1 INCOMPLETE (a check failed)'
rep = {'verdict': VERDICT, 'tests': ALL, 'frozen': FROZEN, 'library_sha256': LIB_SHA, 'events': {k: len(v) for k, v in EV.items()}, 'note': 'synthetic injection; train days injected for detector training (Protocol v3); no test day opened'}
rp = OUT / 'reports' / 'f1_report.json'
if not rp.exists(): rp.write_text(json.dumps(rep, indent=1, default=str))
L = ['==== F1 SHARE BLOCK (copy everything between the two ==== lines) ====', f'{VERDICT} | tests passed {sum(r[2] for r in ALL)}/{len(ALL)} | test days opened: NO | libs {LIB_SHA["stage_e_lib"][:8]} {LIB_SHA["stage_f_lib"][:8]} | scaler {SC_SHA[:12]}', 'events per set: ' + ', '.join(f'{k} {len(v)}' for k, v in EV.items()), 'failed tests: ' + (', '.join(r[0] for r in ALL if not r[2]) or 'none'), '==== END ====']
print('\n'.join(L)); sb = OUT / 'reports' / 'f1_share_block.txt'
if not sb.exists(): sb.write_text('\n'.join(L))

==== F1 SHARE BLOCK (copy everything between the two ==== lines) ====
F1 COMPLETE | tests passed 23/23 | test days opened: NO | libs a67aa38f 375cb175 | scaler 2bfe9d498569
events per set: train_injected/BTCUSDT_2024-05-01 300, train_injected/ETHUSDT_2024-05-01 300, train_injected/BTCUSDT_2024-06-01 300, train_injected/ETHUSDT_2024-06-01 300, val_shift/BTCUSDT_2024-07-01 180, val_shift/ETHUSDT_2024-07-01 180
failed tests: none
==== END ====
